# OSNet-AIN x1.0 — вариант 3: square/letterbox и GeM

Этап сначала фиксирует геометрию входа, затем сравнивает AvgPool и GeM из одного checkpoint на трёх seed. `test_query.csv` и `test_gallery.csv` не используются.

## 1. Окружение и настройки

Полный запуск содержит шесть продолжений обучения: AvgPool и GeM по три seed. Незавершённый запуск продолжается из `last.pt`; завершённый пропускается.

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = Path.cwd().resolve()
while ROOT != ROOT.parent and not (ROOT / 'backend').exists():
    ROOT = ROOT.parent
assert (ROOT / 'backend').exists(), 'Не найден корень Car-classification-MSK'
os.chdir(ROOT)

import torch
from backend.core import DATASET
from training.hpo import experiment_losses, make_optimizer, prepare_experiment
from training.pipeline import ensure_splits, select_device, split_summary
from training.stage3 import (
    compare_preprocessing, config_from_checkpoint, export_pooling_winner,
    initialize_from_checkpoint, run_pooling_ablation,
)

VARIANT_DIR = ROOT / 'OSNet-AIN-x1.0/variant_03_gem'
RESULTS_DIR = VARIANT_DIR / 'results'
WEIGHTS_DIR = VARIANT_DIR / 'weights'
BASE_CHECKPOINT = (ROOT / 'OSNet-AIN-x1.0/variant_02_hpo_bnneck_supcon/'
                   'weights/selected_run_02/best_map.pt')
PREPROCESS_RESULT = RESULTS_DIR / 'preprocessing_ablation.json'
SEEDS = (20260915, 20260916, 20260917)
MAX_EPOCHS = 15
EARLY_STOPPING_PATIENCE = 4
MINIMUM_EPOCHS = 6
RUN_PREPROCESS_ABLATION = False
RUN_POOLING_ABLATION = True
RUN_EXPORT = True
DEVICE = select_device()
assert BASE_CHECKPOINT.exists(), f'Не найден {BASE_CHECKPOINT}'
print('python:', sys.executable)
print('device:', DEVICE, '| torch:', torch.__version__)

python: /Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/bin/python
device: mps | torch: 2.14.0


## 2. Фиксированное разбиение и геометрия

Square и letterbox сравниваются на активном ONNX без обучения. Порог отказа для каждого режима калибруется на calibration; validation используется только для итогового сравнения.

In [2]:
rows, split = ensure_splits(DATASET)
print(json.dumps(split_summary(rows, split), ensure_ascii=False, indent=2))
if RUN_PREPROCESS_ABLATION or not PREPROCESS_RESULT.exists():
    preprocessing = compare_preprocessing(PREPROCESS_RESULT, dataset=DATASET)
else:
    preprocessing = json.loads(PREPROCESS_RESULT.read_text())
for name, result in preprocessing['modes'].items():
    validation = result['reranked']['validation']
    print(name, 'mAP@10=', round(validation['mAP_at_10'], 6),
          'candidate_score=', round(validation['candidate_score'], 6))
RESIZE_MODE = preprocessing['winner']
assert RESIZE_MODE == 'square', 'Перед обучением проверьте неожиданный новый результат геометрии'
print('Выбранная геометрия:', RESIZE_MODE)

{
  "train": {
    "identities": 925,
    "images": 5717
  },
  "calibration": {
    "identities": 307,
    "images": 1903
  },
  "validation": {
    "identities": 309,
    "images": 1936
  }
}
square mAP@10= 0.814689 candidate_score= 0.747121
letterbox mAP@10= 0.66761 candidate_score= 0.669674
Выбранная геометрия: square


## 3. Smoke test GeM

Все веса варианта 2 переносятся в новую модель. Единственный новый параметр — показатель степени `p=3.0` в GeM. Ячейка проверяет полный forward/backward на одном P×K batch.

In [3]:
smoke_config = config_from_checkpoint(BASE_CHECKPOINT, 'gem', RESIZE_MODE, SEEDS[0], 1)
smoke_rows, smoke_labels, smoke_sampler, smoke_loader = prepare_experiment(
    rows, split['identities']['train'], smoke_config, DATASET)
smoke_model, classes = initialize_from_checkpoint(BASE_CHECKPOINT, smoke_config, DEVICE)
clean, robust, labels, _ = next(iter(smoke_loader))
optimizer = make_optimizer(smoke_model, smoke_config)
optimizer.zero_grad(set_to_none=True)
losses = experiment_losses(
    smoke_model, clean.to(DEVICE), robust.to(DEVICE), labels.to(DEVICE), smoke_config)
losses['loss'].backward(); optimizer.step()
assert classes == len(smoke_labels) == len(split['identities']['train'])
assert torch.isfinite(smoke_model.backbone.global_pool.p.grad)
print('loss:', float(losses['loss']), '| GeM p:', float(smoke_model.backbone.global_pool.p))
del smoke_model, smoke_loader, optimizer, clean, robust, labels, losses
if DEVICE.type == 'mps': torch.mps.empty_cache()
if DEVICE.type == 'cuda': torch.cuda.empty_cache()

loss: 1.918613076210022 | GeM p: 2.99988055229187


/var/folders/ln/cmj3zx5n3b7bk0hqwh3vwsqc0000gn/T/ipykernel_97799/1406559688.py:13: UserWarning: Converting a tensor with requires_grad=True to a scalar may lead to unexpected behavior.
Consider using tensor.detach() first. (Triggered internally at /Users/runner/work/pytorch/pytorch/torch/csrc/autograd/generated/python_variable_methods.cpp:821.)
  print('loss:', float(losses['loss']), '| GeM p:', float(smoke_model.backbone.global_pool.p))


## 4. AvgPool против GeM на трёх seed

Контроль обязателен: без него рост после дополнительных эпох можно ошибочно приписать GeM. Early stopping начинается не раньше шестой эпохи. Raw-best checkpoint каждого seed дополнительно оценивается с активным реранкингом.

In [4]:
if RUN_POOLING_ABLATION:
    pooling_report = run_pooling_ablation(
        rows, split, BASE_CHECKPOINT, DEVICE, RESULTS_DIR, WEIGHTS_DIR,
        resize_mode=RESIZE_MODE, seeds=SEEDS, epochs=MAX_EPOCHS,
        patience=EARLY_STOPPING_PATIENCE, minimum_epochs=MINIMUM_EPOCHS, dataset=DATASET)
else:
    pooling_report = json.loads((RESULTS_DIR / 'pooling_ablation.json').read_text())
print('winner:', pooling_report['winner'])
for name, result in pooling_report['aggregates'].items():
    print(name, 'mean mAP=', round(result['mean_best_mAP'], 6),
          'std=', round(result['std_best_mAP'], 6),
          'candidate=', round(result['mean_candidate_score_at_best_mAP'], 6))

epoch 1:   0%|          | 0/57 [00:00<?, ?it/s]

Epoch 1/15 | осталось эпох: 14 | эпоха: 00:01:16 | прошло: 00:01:16 | ETA: 00:17:42 | best mAP: 0.7845


epoch 2:   0%|          | 0/57 [00:00<?, ?it/s]

Epoch 2/15 | осталось эпох: 13 | эпоха: 00:01:15 | прошло: 00:02:32 | ETA: 00:16:29 | best mAP: 0.7845


epoch 3:   0%|          | 0/57 [00:00<?, ?it/s]

...

Epoch 5/15 | осталось эпох: 10 | эпоха: 00:01:14 | прошло: 00:06:14 | ETA: 00:12:28 | best mAP: 0.7883


epoch 6:   0%|          | 0/57 [00:00<?, ?it/s]

Epoch 6/15 | осталось эпох: 9 | эпоха: 00:01:14 | прошло: 00:07:28 | ETA: 00:11:12 | best mAP: 0.7883


epoch 7:   0%|          | 0/57 [00:00<?, ?it/s]

Epoch 7/15 | осталось эпох: 8 | эпоха: 00:01:14 | прошло: 00:08:43 | ETA: 00:09:58 | best mAP: 0.7883
winner: avg
avg mean mAP= 0.80025 std= 0.005829 candidate= 0.757405
gem mean mAP= 0.79111 std= 0.007347 candidate= 0.733111


## 5. Экспорт репрезентативного запуска

Экспортируется seed, результат которого ближе всего к среднему победившей группы, а не самый удачный seed. PyTorch/ONNX parity проверяется на реальном BBox-кропе. ONNX не подключается к MVP автоматически.

In [5]:
if RUN_EXPORT:
    export_summary = export_pooling_winner(
        pooling_report, rows, split, WEIGHTS_DIR,
        WEIGHTS_DIR / 'osnet_stage3_selected.onnx',
        RESULTS_DIR / 'export_summary.json', DATASET)
    print(json.dumps(export_summary, ensure_ascii=False, indent=2))

/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/.venv/lib/python3.11/site-packages/torch/onnx/_internal/torchscript_exporter/symbolic_opset9.py:2855: UserWarning: ONNX export mode is set to TrainingMode.EVAL, but operator 'instance_norm' is set to train=True. Exporting with train=True.
  symbolic_helper.check_training_mode(use_input_stats, "instance_norm")


{
  "checkpoint": "/Users/elvsevolod/Desktop/учеба/Учёба 4 курс/Хакатон мск сентябрь/Car-classification-MSK/OSNet-AIN-x1.0/variant_03_gem/weights/avg_seed_20260915/best_map.pt",
  "checkpoint_epoch": 5,
  "config": {
    "epochs": 15,
    "identities_per_batch": 16,
    "images_per_identity": 2,
    "encoder_lr": 0.00011940564013110387,
    "head_lr_multiplier": 10.0,
    "weight_decay": 6.069870050850335e-05,
    "warmup_epochs": 2,
    "min_lr_ratio": 0.02,
    "metric_loss": "supcon",
    "metric_weight": 1.4948762510958067,
    "triplet_margin": 0.5,
    "supcon_temperature": 0.1,
    "consistency_weight": 0.22571745222502657,
    "label_smoothing": 0.1,
    "use_bnneck": true,
    "prefer_cross_camera": true,
    "num_workers": 0,
    "seed": 20260915,
    "pooling": "avg",
    "resize_mode": "square"
  },
  "threshold": 0.5753594040870667,
  "calibration": {
    "mAP": 0.7739098593366885,
    "mAP_at_10": 0.7739098593366885,
    "full_mAP": 0.7885865623247814,
    "Rank_1": 0.79

## 6. После обучения

Для анализа нужны `pooling_ablation.json`, `export_summary.json` и шесть `summary.json`. Только после сравнения среднего, разброса и open-set score решаем, менять ли активный MVP.